# Steam Review NLP Data Analysis\n\nThis Colab notebook analyzes Steam user review text using NLP. It loads the two CSV files, cleans review text, explores sentiment patterns, trains a TF-IDF sentiment classifier, and exports tables/figures/model files.

## Data Setup\n\nUse one of these options:\n\n**Option A: Upload files directly to Colab**\n- Upload `output.csv`\n- Upload `output_steamspy.csv`\n\n**Option B: Use Google Drive**\n- Put the folder in `MyDrive/Developer_Remote/`\n- Set `USE_GOOGLE_DRIVE = True` in the next cell\n\nExpected files:\n- `output.csv`: review data with `content` and `is_positive`\n- `output_steamspy.csv`: game metadata with `appid`, `name`, and `owners`

In [ ]:
from pathlib import Path\nimport re\nimport zipfile\nfrom html import unescape\n\nimport joblib\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, roc_auc_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.pipeline import Pipeline\n\nUSE_GOOGLE_DRIVE = False\nSAMPLE_SIZE = None  # Set to 100000 for a faster test run, or None for the full dataset.\nRANDOM_STATE = 42\n\nif USE_GOOGLE_DRIVE:\n    from google.colab import drive\n    drive.mount('/content/drive')\n\nOUTPUT_DIR = Path('/content/steam_nlp_outputs')\nTABLES_DIR = OUTPUT_DIR / 'tables'\nFIGURES_DIR = OUTPUT_DIR / 'figures'\nMODELS_DIR = OUTPUT_DIR / 'models'\nfor folder in [TABLES_DIR, FIGURES_DIR, MODELS_DIR]:\n    folder.mkdir(parents=True, exist_ok=True)\n\nsns.set_theme(style='whitegrid')\npd.set_option('display.max_colwidth', 120)\nprint('Output folder:', OUTPUT_DIR)

In [ ]:
def find_data_file(filename):\n    search_roots = [\n        Path('/content'),\n        Path('/content/Steam Review&Games Dataset'),\n        Path('/content/Developer_Remote'),\n        Path('/content/Developer_Remote/Steam Review&Games Dataset'),\n        Path('/content/drive/MyDrive/Developer_Remote'),\n        Path('/content/drive/MyDrive/Developer_Remote/Steam Review&Games Dataset'),\n    ]\n    for root in search_roots:\n        candidate = root / filename\n        if candidate.exists():\n            return candidate\n    matches = list(Path('/content').rglob(filename))\n    return matches[0] if matches else None\n\nreviews_path = find_data_file('output.csv')\ngames_path = find_data_file('output_steamspy.csv')\n\nif reviews_path is None or games_path is None:\n    print('Upload output.csv and output_steamspy.csv now.')\n    from google.colab import files\n    files.upload()\n    reviews_path = find_data_file('output.csv')\n    games_path = find_data_file('output_steamspy.csv')\n\nif reviews_path is None or games_path is None:\n    raise FileNotFoundError('Could not find both output.csv and output_steamspy.csv.')\n\nprint('Reviews:', reviews_path)\nprint('Games:', games_path)

In [ ]:
reviews = pd.read_csv(reviews_path)\ngames = pd.read_csv(games_path)\n\nif SAMPLE_SIZE is not None and len(reviews) > SAMPLE_SIZE:\n    reviews = reviews.sample(SAMPLE_SIZE, random_state=RANDOM_STATE).reset_index(drop=True)\n\nprint('Reviews shape:', reviews.shape)\nprint('Games shape:', games.shape)\ndisplay(reviews.head())\ndisplay(games.head())

In [ ]:
URL_RE = re.compile(r'https?://\\S+|www\\.\\S+')\nHTML_TAG_RE = re.compile(r'<[^>]+>')\nWHITESPACE_RE = re.compile(r'\\s+')\n\ndef clean_review_text(value):\n    text = '' if pd.isna(value) else str(value)\n    text = unescape(text)\n    text = HTML_TAG_RE.sub(' ', text)\n    text = URL_RE.sub(' ', text)\n    text = text.replace('\\r', ' ').replace('\\n', ' ')\n    text = WHITESPACE_RE.sub(' ', text)\n    return text.strip().lower()\n\ndf = reviews.copy()\ndf['label'] = df['is_positive'].map({'Positive': 1, 'Negative': 0})\ndf['clean_text'] = df['content'].map(clean_review_text)\ndf['review_length_chars'] = df['clean_text'].str.len()\ndf['review_length_words'] = df['clean_text'].str.split().str.len()\n\ngames_clean = games.rename(columns={'appid': 'app_id', 'name': 'game_name'})\ndf = df.merge(games_clean, on='app_id', how='left')\ndf['game_name'] = df['game_name'].fillna('Unknown')\ndf['owners'] = df['owners'].fillna('Unknown')\ndf = df.dropna(subset=['label', 'clean_text'])\ndf = df[df['clean_text'].str.len() > 0].copy()\ndf['label'] = df['label'].astype(int)\n\nclean_path = OUTPUT_DIR / 'clean_reviews.csv'\ndf.to_csv(clean_path, index=False)\nprint('Clean rows:', f'{len(df):,}')\ndisplay(df[['app_id', 'game_name', 'is_positive', 'review_length_words', 'clean_text']].head())

In [ ]:
summary = pd.DataFrame([\n    {'metric': 'reviews', 'value': len(df)},\n    {'metric': 'unique_games', 'value': df['app_id'].nunique()},\n    {'metric': 'unique_authors', 'value': df['author_id'].nunique()},\n    {'metric': 'positive_reviews', 'value': int(df['label'].sum())},\n    {'metric': 'negative_reviews', 'value': int((1 - df['label']).sum())},\n    {'metric': 'positive_rate', 'value': round(float(df['label'].mean()), 4)},\n    {'metric': 'median_review_words', 'value': round(float(df['review_length_words'].median()), 2)},\n    {'metric': 'mean_review_words', 'value': round(float(df['review_length_words'].mean()), 2)},\n])\n\ntop_games = (\n    df.groupby(['app_id', 'game_name'], dropna=False)\n      .agg(review_count=('id', 'count'), positive_rate=('label', 'mean'), median_words=('review_length_words', 'median'))\n      .reset_index()\n      .sort_values('review_count', ascending=False)\n)\ntop_games['positive_rate'] = top_games['positive_rate'].round(4)\ntop_games['median_words'] = top_games['median_words'].round(2)\n\nsummary.to_csv(TABLES_DIR / 'dataset_summary.csv', index=False)\ntop_games.head(50).to_csv(TABLES_DIR / 'top_games.csv', index=False)\n\ndisplay(summary)\ndisplay(top_games.head(15))

In [ ]:
plt.figure(figsize=(7, 4))\nax = sns.countplot(data=df, x='is_positive', order=['Negative', 'Positive'])\nax.set_title('Steam Review Label Distribution')\nax.set_xlabel('Review Label')\nax.set_ylabel('Review Count')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'label_distribution.png', dpi=160)\nplt.show()\n\nplt.figure(figsize=(8, 4))\nclipped = df[df['review_length_words'].between(1, df['review_length_words'].quantile(0.99))]\nax = sns.histplot(data=clipped, x='review_length_words', hue='is_positive', bins=50)\nax.set_title('Review Length Distribution')\nax.set_xlabel('Words per Review')\nax.set_ylabel('Review Count')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'review_length_distribution.png', dpi=160)\nplt.show()\n\nplot_games = top_games.head(15).sort_values('review_count', ascending=True)\nplt.figure(figsize=(9, 6))\nax = sns.barplot(data=plot_games, x='review_count', y='game_name', color='#4C78A8')\nax.set_title('Top Games by Review Count')\nax.set_xlabel('Review Count')\nax.set_ylabel('')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'top_games_by_review_count.png', dpi=160)\nplt.show()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(\n    df['clean_text'],\n    df['label'],\n    test_size=0.2,\n    random_state=RANDOM_STATE,\n    stratify=df['label'],\n)\n\npipeline = Pipeline([\n    ('tfidf', TfidfVectorizer(\n        max_features=30000,\n        min_df=3,\n        max_df=0.9,\n        ngram_range=(1, 2),\n        stop_words='english',\n        sublinear_tf=True,\n    )),\n    ('classifier', LogisticRegression(\n        max_iter=1000,\n        class_weight='balanced',\n        random_state=RANDOM_STATE,\n    )),\n])\n\npipeline.fit(x_train, y_train)\npredictions = pipeline.predict(x_test)\nprobabilities = pipeline.predict_proba(x_test)[:, 1]\n\nmetrics = pd.DataFrame([\n    {'metric': 'accuracy', 'value': accuracy_score(y_test, predictions)},\n    {'metric': 'f1', 'value': f1_score(y_test, predictions)},\n    {'metric': 'roc_auc', 'value': roc_auc_score(y_test, probabilities)},\n])\nmetrics['value'] = metrics['value'].round(4)\n\nreport = pd.DataFrame(classification_report(y_test, predictions, output_dict=True)).T\ncm = confusion_matrix(y_test, predictions)\n\nmetrics.to_csv(TABLES_DIR / 'model_metrics.csv', index=False)\nreport.to_csv(TABLES_DIR / 'classification_report.csv')\njoblib.dump(pipeline, MODELS_DIR / 'tfidf_logistic_regression.joblib')\n\ndisplay(metrics)\ndisplay(report.round(4))\n\nplt.figure(figsize=(5, 4))\nax = sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Negative', 'Positive'], yticklabels=['Negative', 'Positive'])\nax.set_title('Confusion Matrix')\nax.set_xlabel('Predicted')\nax.set_ylabel('Actual')\nplt.tight_layout()\nplt.savefig(FIGURES_DIR / 'confusion_matrix.png', dpi=160)\nplt.show()

In [ ]:
vectorizer = pipeline.named_steps['tfidf']\nclassifier = pipeline.named_steps['classifier']\nterms = pd.DataFrame({\n    'term': vectorizer.get_feature_names_out(),\n    'coefficient': classifier.coef_[0],\n})\n\ntop_positive_terms = terms.sort_values('coefficient', ascending=False).head(30).reset_index(drop=True)\ntop_negative_terms = terms.sort_values('coefficient', ascending=True).head(30).reset_index(drop=True)\n\ntop_positive_terms.to_csv(TABLES_DIR / 'top_positive_terms.csv', index=False)\ntop_negative_terms.to_csv(TABLES_DIR / 'top_negative_terms.csv', index=False)\n\nprint('Top terms associated with positive reviews')\ndisplay(top_positive_terms)\nprint('Top terms associated with negative reviews')\ndisplay(top_negative_terms)

In [ ]:
examples = pd.Series([\n    'Great combat, smooth gameplay, and I would absolutely recommend it.',\n    'Crashes constantly, terrible controls, and not fun at all.',\n])\nexample_scores = pipeline.predict_proba(examples)[:, 1]\npd.DataFrame({'review': examples, 'positive_probability': example_scores.round(4)})

In [ ]:
zip_path = Path('/content/steam_nlp_outputs.zip')\nwith zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:\n    for path in OUTPUT_DIR.rglob('*'):\n        if path.is_file():\n            zf.write(path, path.relative_to(OUTPUT_DIR.parent))\n\nprint('Created:', zip_path)\nfrom google.colab import files\nfiles.download(str(zip_path))